# iPLS for Maize MIR Regression


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import KFold
from sklearn.base import clone

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)


## Load data

In [ ]:
X_raw = np.loadtxt("dataF_final.csv", delimiter=",")
Y_raw = np.loadtxt("dataC_final.csv", delimiter=",")

if Y_raw.ndim == 1:
    Y_raw = Y_raw.reshape(-1, 1)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X = scaler_x.fit_transform(X_raw)
Y = scaler_y.fit_transform(Y_raw)

print("X shape:", X.shape)
print("Y shape:", Y.shape)


X shape: (258, 1868)
Y shape: (258, 5)


## Helper functions

In [ ]:
def make_intervals(n_features, n_intervals):
    """Split wavelengths into contiguous iPLS intervals."""
    return [arr for arr in np.array_split(np.arange(n_features), n_intervals) if len(arr) > 0]


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def cv_predict_and_score(X_subset, y, model_template):
    """5-fold CV prediction and score"""
    y_true_all = []
    y_pred_all = []
    fold_r2 = []
    fold_rmse = []

    for train_idx, test_idx in kf.split(X_subset):
        X_train, X_test = X_subset[train_idx], X_subset[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = clone(model_template)
        if isinstance(model, PLSRegression):
            model.n_components = min(model.n_components, X_train.shape[1], len(train_idx) - 1)

        model.fit(X_train, y_train)
        pred = model.predict(X_test).ravel()

        y_true_all.extend(y_test)
        y_pred_all.extend(pred)
        fold_r2.append(r2_score(y_test, pred))
        fold_rmse.append(rmse(y_test, pred))

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)

    return {
        "R2_test": float(np.mean(fold_r2)),
        "RMSE_test": float(np.mean(fold_rmse)),
        "R2_all_folds": float(r2_score(y_true_all, y_pred_all)),
        "RMSE_all_folds": float(rmse(y_true_all, y_pred_all)),
        "y_true": y_true_all,
        "y_pred": y_pred_all
    }


def best_plsr_score(X_subset, y, component_grid=(1, 2, 3, 5, 8, 10, 12, 15, 20, 25, 30)):
    best = None
    max_comp_allowed = min(X_subset.shape[1], X_subset.shape[0] - 1)

    for n_comp in component_grid:
        if n_comp > max_comp_allowed:
            continue
        result = cv_predict_and_score(X_subset, y, PLSRegression(n_components=n_comp))
        if best is None or result["R2_all_folds"] > best["R2_all_folds"]:
            best = result.copy()
            best["n_components"] = n_comp

    return best


def interval_correlation_ranking(X, y, intervals):
    y_centered = y - y.mean()
    y_std = y_centered.std() + 1e-12
    scores = []

    for i, idx in enumerate(intervals):
        Xi = X[:, idx]
        Xi_centered = Xi - Xi.mean(axis=0)
        corr = np.abs((Xi_centered.T @ y_centered) / ((len(y) - 1) * (Xi.std(axis=0) + 1e-12) * y_std))
        score = np.mean(np.sort(corr)[-min(10, len(corr)):])
        scores.append((score, i))

    return [i for _, i in sorted(scores, reverse=True)]


## iPLS feature selection

In [ ]:
def optimized_ipls_selection(
    X,
    y,
    interval_options=(8, 10, 12, 15, 20, 25, 30, 35, 40, 50, 60),
    top_pool=18,
    max_forward_intervals=10,
    top_k_values=(1, 2, 3, 4, 5, 6, 8, 10, 12, 15),
    component_grid=(1, 2, 3, 5, 8, 10, 12, 15, 20, 25, 30)
):
   
    global_best = None
    global_best_idx = None

    for n_intervals in interval_options:
        intervals = make_intervals(X.shape[1], n_intervals)
        ranked = interval_correlation_ranking(X, y, intervals)
        ranked = ranked[:min(top_pool, len(ranked))]

        candidate_sets = []

        for k in top_k_values:
            k = min(k, len(ranked))
            if k > 0:
                candidate_sets.append(sorted(ranked[:k]))

        selected = []
        best_forward_score = -np.inf

        for step in range(min(max_forward_intervals, len(ranked))):
            step_best = None

            for candidate in ranked:
                if candidate in selected:
                    continue

                trial_intervals = sorted(selected + [candidate])
                trial_idx = np.unique(np.concatenate([intervals[i] for i in trial_intervals]))
                trial_score = best_plsr_score(X[:, trial_idx], y, component_grid=component_grid)["R2_all_folds"]

                if step_best is None or trial_score > step_best[0]:
                    step_best = (trial_score, candidate, trial_intervals)

            if step_best is None:
                break

            if step == 0 or step_best[0] > best_forward_score + 1e-5:
                best_forward_score, chosen_interval, selected = step_best
                candidate_sets.append(selected.copy())
            else:
                break

        expanded_sets = []
        for selected_set in candidate_sets:
            expanded = set(selected_set)
            for j in selected_set:
                if j - 1 >= 0:
                    expanded.add(j - 1)
                if j + 1 < len(intervals):
                    expanded.add(j + 1)
            expanded_sets.append(sorted(expanded))

        candidate_sets.extend(expanded_sets)

        unique_candidate_sets = []
        seen = set()
        for s in candidate_sets:
            key = tuple(s)
            if key not in seen:
                unique_candidate_sets.append(s)
                seen.add(key)

        for selected_intervals in unique_candidate_sets:
            selected_idx = np.unique(np.concatenate([intervals[i] for i in selected_intervals]))
            score = best_plsr_score(X[:, selected_idx], y, component_grid=component_grid)

            info = {
                "n_intervals": n_intervals,
                "selected_intervals": selected_intervals,
                "features": len(selected_idx),
                "selection_R2_all_folds": score["R2_all_folds"],
                "selection_RMSE_all_folds": score["RMSE_all_folds"],
                "selection_n_components": score["n_components"]
            }

            if global_best is None or info["selection_R2_all_folds"] > global_best["selection_R2_all_folds"]:
                global_best = info
                global_best_idx = selected_idx

    return np.sort(global_best_idx), global_best


## Tuned final model evaluation

In [ ]:
def get_model_grid(n_features):
    model_grid = []

    for n_comp in [1, 2, 3, 5, 8, 10, 12, 15, 20, 25, 30, 40]:
        if n_comp <= n_features:
            model_grid.append(("PLSR", {"n_components": n_comp}, PLSRegression(n_components=n_comp)))

    for alpha in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000]:
        model_grid.append(("Ridge", {"alpha": alpha}, Ridge(alpha=alpha)))

    for alpha in [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]:
        model_grid.append(("Lasso", {"alpha": alpha}, Lasso(alpha=alpha, max_iter=50000)))

    for alpha in [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]:
        for l1_ratio in [0.001, 0.01, 0.05, 0.1, 0.3, 0.5, 0.8]:
            model_grid.append(("ElasticNet", {"alpha": alpha, "l1_ratio": l1_ratio}, ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=50000)))

    return model_grid


def evaluate_tuned_models(X_selected, y_target, target_number, method_name="Optimized iPLS"):
    rows = []
    predictions = {}

    for model_name in ["PLSR", "Ridge", "Lasso", "ElasticNet"]:
        best_result = None
        best_params = None

        for name, params, model in get_model_grid(X_selected.shape[1]):
            if name != model_name:
                continue

            result = cv_predict_and_score(X_selected, y_target, model)

            if best_result is None or result["R2_all_folds"] > best_result["R2_all_folds"]:
                best_result = result
                best_params = params

        rows.append({
            "Method": method_name,
            "Target": target_number,
            "Model": model_name,
            "Best_Params": str(best_params),
            "R2_test": best_result["R2_test"],
            "RMSE_test": best_result["RMSE_test"],
            "R2_all_folds": best_result["R2_all_folds"],
            "RMSE_all_folds": best_result["RMSE_all_folds"],
            "Features": X_selected.shape[1]
        })

        predictions[(target_number, model_name)] = (best_result["y_true"], best_result["y_pred"])

    return rows, predictions


## Run iPLS

In [ ]:
results_list_ipls_optimized = []
all_predictions_ipls_optimized = {}
selected_features_ipls_optimized = {}
ipls_optimized_info = {}

print(f"{'Target':<8} | {'Model':<12} | {'R2_test':<10} | {'RMSE_test':<10} | {'R2_all':<10} | {'Features':<8} | Params")
print("-" * 125)

for t in range(Y.shape[1]):
    print(f"Optimizing Target {t+1} with optimized iPLS...")
    start = time.time()

    y_target = Y[:, t]

    best_idx, info = optimized_ipls_selection(
        X,
        y_target,
        interval_options=(8, 10, 12, 15, 20, 25, 30, 35, 40, 50, 60),
        top_pool=18,
        max_forward_intervals=10
    )

    X_selected = X[:, best_idx]
    selected_features_ipls_optimized[t + 1] = best_idx
    ipls_optimized_info[t + 1] = info

    rows, preds = evaluate_tuned_models(X_selected, y_target, t + 1, "Optimized iPLS")
    results_list_ipls_optimized.extend(rows)
    all_predictions_ipls_optimized.update(preds)

    for row in rows:
        print(f"T{row['Target']:<6} | {row['Model']:<12} | {row['R2_test']:.4f}   | {row['RMSE_test']:.4f}   | {row['R2_all_folds']:.4f}   | {row['Features']:<8} | {row['Best_Params']}")

    print("Optimized iPLS info:", info)
    print(f"RunTime: {time.time() - start:.2f}s")

results_df_ipls_optimized = pd.DataFrame(results_list_ipls_optimized)
results_df_ipls_optimized.to_csv("Optimized_IPLS_results.csv", index=False)
results_df_ipls_optimized


Target   | Model        | R2_test    | RMSE_test  | R2_all     | Features | Params
-----------------------------------------------------------------------------------------------------------------------------
Optimizing Target 1 with optimized iPLS...
T1      | PLSR         | 0.3393   | 0.8012   | 0.3517   | 155      | {'n_components': 5}
T1      | Ridge        | 0.3322   | 0.8066   | 0.3428   | 155      | {'alpha': 30}
T1      | Lasso        | 0.3198   | 0.8150   | 0.3287   | 155      | {'alpha': 0.03}
T1      | ElasticNet   | 0.3346   | 0.8052   | 0.3449   | 155      | {'alpha': 0.1, 'l1_ratio': 0.05}
Optimized iPLS info: {'n_intervals': 60, 'selected_intervals': [23, 27, 31, 32, 58], 'features': 155, 'selection_R2_all_folds': 0.35166202115747536, 'selection_RMSE_all_folds': 0.8051943733301449, 'selection_n_components': 5}
RunTime: 695.43s
Optimizing Target 2 with optimized iPLS...
T2      | PLSR         | 0.4248   | 0.7554   | 0.4247   | 155      | {'n_components': 5}
T2      | Ridg

,Method,Target,Model,Best_Params,R2_test,RMSE_test,R2_all_folds,RMSE_all_folds,Features
0,Optimized iPLS,1,PLSR,{'n_components': 5},0.339316,0.801214,0.351662,0.805194,155
1,Optimized iPLS,1,Ridge,{'alpha': 30},0.332223,0.806628,0.342754,0.810707,155
2,Optimized iPLS,1,Lasso,{'alpha': 0.03},0.319767,0.815029,0.328695,0.819332,155
3,Optimized iPLS,1,ElasticNet,"{'alpha': 0.1, 'l1_ratio': 0.05}",0.334647,0.805245,0.344941,0.809357,155
4,Optimized iPLS,2,PLSR,{'n_components': 5},0.424839,0.755403,0.424744,0.758456,155
5,Optimized iPLS,2,Ridge,{'alpha': 100},0.427040,0.753844,0.426039,0.757602,155
6,Optimized iPLS,2,Lasso,{'alpha': 0.03},0.418864,0.759178,0.419051,0.762200,155
7,Optimized iPLS,2,ElasticNet,"{'alpha': 0.1, 'l1_ratio': 0.1}",0.425556,0.754661,0.426734,0.757144,155
8,Optimized iPLS,3,PLSR,{'n_components': 5},0.302003,0.774239,0.389360,0.781435,310
9,Optimized iPLS,3,Ridge,{'alpha': 100},0.340527,0.766116,0.395724,0.777352,310


## Best model per target

In [ ]:
best_models_ipls_optimized = (
    results_df_ipls_optimized.sort_values("R2_all_folds", ascending=False)
    .groupby("Target")
    .first()
    .reset_index()
)

best_models_ipls_optimized.to_csv("Optimized_IPLS_best_models.csv", index=False)
best_models_ipls_optimized


,Target,Method,Model,Best_Params,R2_test,RMSE_test,R2_all_folds,RMSE_all_folds,Features
0,1,Optimized iPLS,PLSR,{'n_components': 5},0.339316,0.801214,0.351662,0.805194,155
1,2,Optimized iPLS,ElasticNet,"{'alpha': 0.1, 'l1_ratio': 0.1}",0.425556,0.754661,0.426734,0.757144,155
2,3,Optimized iPLS,Ridge,{'alpha': 100},0.340527,0.766116,0.395724,0.777352,310
3,4,Optimized iPLS,ElasticNet,"{'alpha': 0.1, 'l1_ratio': 0.5}",0.392826,0.773009,0.383418,0.785228,746
4,5,Optimized iPLS,Ridge,{'alpha': 300},0.031743,0.924528,0.039534,0.980034,186


## Save selected features and iPLS details

In [ ]:
import json

selected_features_to_save = {str(k): v.tolist() for k, v in selected_features_ipls_optimized.items()}

with open("Optimized_IPLS_selected_features.json", "w") as f:
    json.dump(selected_features_to_save, f, indent=2)

with open("Optimized_IPLS_selection_info.json", "w") as f:
    json.dump(ipls_optimized_info, f, indent=2)

print("Saved:")
print("- Optimized_IPLS_results.csv")
print("- Optimized_IPLS_best_models.csv")
print("- Optimized_IPLS_selected_features.json")
print("- Optimized_IPLS_selection_info.json")


NameError: name 'selected_features_ipls_optimized' is not defined

## Plot best predicted vs observed results

In [3]:
def plot_best_models(best_models, all_predictions):
    cols = 3
    rows = int(np.ceil(len(best_models) / cols))
    plt.figure(figsize=(14, 4 * rows))

    for i, row in best_models.iterrows():
        t = row["Target"]
        model = row["Model"]
        y_true, y_pred = all_predictions[(t, model)]

        rmse_value = rmse(y_true, y_pred)
        r2_value = r2_score(y_true, y_pred)

        plt.subplot(rows, cols, i + 1)
        plt.scatter(y_true, y_pred, s=12)

        min_v = min(y_true.min(), y_pred.min())
        max_v = max(y_true.max(), y_pred.max())
        plt.plot([min_v, max_v], [min_v, max_v], "r--")

        plt.title(f"T{t} - {model}RMSE={rmse_value:.3f}, R²={r2_value:.3f}")
        plt.xlabel("Observed")
        plt.ylabel("Predicted")
        plt.grid()

    plt.tight_layout()
    plt.show()

plot_best_models(best_models_ipls_optimized, all_predictions_ipls_optimized)


NameError: name 'best_models_ipls_optimized' is not defined